In [ ]:
import MetaTrader5 as mt5
from datetime import datetime
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
import optuna
import numpy as np
import pandas as pd
import yfinance as yf
from stable_baselines3.common.evaluation import evaluate_policy
import gymnasium as gym
from gymnasium import spaces
import gc
from wandb.integration.sb3 import WandbCallback
import wandb
from alibi_detect.od import IForest
from stable_baselines3.common.callbacks import BaseCallback
import torch
import os
from stable_baselines3.common.callbacks import EvalCallback
import matplotlib.pyplot as plt
import numpy as np
import random
from stable_baselines3.common.vec_env import VecEnvWrapper

d:\dev python\OrionTrader\.venv\lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
d:\dev python\OrionTrader\.venv\lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type ali

## Config

In [8]:
MT5_LOGIN = os.environ.get('MT5_LOGIN')        # ex: 12345678 ou None pour pas d'auth (local)
MT5_PASSWORD = os.environ.get('MT5_PASSWORD')      # placeholder
MT5_SERVER = os.environ.get('MT5_SERVER')     # ex: "MetaQuotes-Demo"
SYMBOL = "EURUSD"       # ajuster
TIMEFRAME = mt5.TIMEFRAME_D1
START = datetime(2020,1,1)
END = datetime(2023,1,1)
MODEL_PATH = "models/best_ppo_forex.zip"  # chemin du modèle sauvegardé
RESUME_TIMESTEPS = 200_000
MT5_PATH = r"C:\Users\Aurelien\AppData\Roaming\MetaTrader 5\terminal64.exe"

## Connexion MT5 et récupération historique

In [9]:
import MetaTrader5 as mt5
import time
import platform
import os

def connect_mt5(max_retries=3, wait_time=2):
    """
    Initialise la connexion à MetaTrader 5 avec vérifications et logs détaillés.
    """
    print("🔹 Fermeture d'éventuelles connexions précédentes...")
    mt5.shutdown()
    time.sleep(1)

    for attempt in range(1, max_retries + 1):
        print(f"\n🔄 Tentative {attempt}/{max_retries} d'initialisation de MetaTrader 5...")

        # Essayer d'initialiser avec chemin explicite
        if not mt5.initialize(path=MT5_PATH):
            err = mt5.last_error()
            print(f"❌ Échec de l'initialisation : {err}")
            time.sleep(wait_time)
            continue

        # Vérifier la version / architecture
        mt5_version = mt5.version()
        print(f"✅ MetaTrader 5 initialisé : version {mt5_version}")
        print(f"💻 Python : {platform.architecture()[0]} | MT5 : {'64-bit' if mt5_version[1] == 64 else '32-bit'}")

        # Connexion au compte
        if MT5_LOGIN:
            print(f"🔐 Connexion au compte {MT5_LOGIN} ({MT5_SERVER})...")
            if not mt5.login(login=MT5_LOGIN, password=MT5_PASSWORD, server=MT5_SERVER):
                err = mt5.last_error()
                print(f"❌ Connexion échouée : {err}")
                mt5.shutdown()
                time.sleep(wait_time)
                continue
            print("✅ Connexion réussie à MetaTrader 5.")
        else:
            print("⚠️ Aucun identifiant de compte spécifié, MT5 connecté sans login.")

        # Si on arrive ici : succès
        return True

    print("⛔ Impossible d'initialiser MetaTrader 5 après plusieurs tentatives.")
    return False



if connect_mt5():
    print("\n📈 Connexion réussie. Récupération de données de test...")
    rates = mt5.copy_rates_from_pos("EURUSD", mt5.TIMEFRAME_M1, 0, 10)
    if rates is not None and len(rates) > 0:
        print("✅ Données récupérées :")
        print(rates[:3])
    else:
        print("⚠️ Aucune donnée récupérée.")
else:
        print("❌ Connexion impossible.")

🔹 Fermeture d'éventuelles connexions précédentes...

🔄 Tentative 1/3 d'initialisation de MetaTrader 5...
❌ Échec de l'initialisation : (-10005, 'IPC timeout')

🔄 Tentative 2/3 d'initialisation de MetaTrader 5...
❌ Échec de l'initialisation : (-10005, 'IPC timeout')

🔄 Tentative 3/3 d'initialisation de MetaTrader 5...
❌ Échec de l'initialisation : (-10005, 'IPC timeout')
⛔ Impossible d'initialiser MetaTrader 5 après plusieurs tentatives.
❌ Connexion impossible.


## Créer env compatible déjà utilisé dans notebook

In [ ]:
class ForexEnv(gym.Env):
    def __init__(self, df):
        super(ForexEnv, self).__init__()
        self.df = df.reset_index(drop=True)
        self.max_steps = len(df) - 1
        self.current_step = 0

        # Action : 0 = hold, 1 = buy, 2 = sell
        self.action_space = spaces.Discrete(3)

        # Observation : prix actuel et delta précédent
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(2,), dtype=np.float32)

        self.position = 0  # 0 = neutral, 1 = long, -1 = short
        self.entry_price = 0.0

    def reset(self, **kwargs):
        self.current_step = 0
        self.position = 0
        self.entry_price = 0.0
        obs = self._next_obs()
        info = {}
        return obs, info

    def _next_obs(self):
        price = float(self.df.loc[self.current_step, 'Close'])
        delta = float(self.df.loc[self.current_step, 'Close'] - self.df.loc[self.current_step-1, 'Close']) if self.current_step > 0 else 0.0
        obs = np.array([price, delta], dtype=np.float32)
        return obs
    
    def step(self, action):
        prev_price = float(self.df.loc[self.current_step, 'Close'])
        self.current_step += 1
        terminated = self.current_step >= len(self.df) - 1
        price = float(self.df.loc[self.current_step, 'Close'])
        delta = price - prev_price

        # Actions : 0 hold, 1 buy, 2 sell, 3 close
        if action == 1:
            self.position = 1
        elif action == 2:
            self.position = -1
        elif action == 3:
            self.position = 0

        reward = delta * self.position * 1000  # amplifié pour stabilité

        obs = np.array([price, delta], dtype=np.float32)
        info = {}
        truncated = False
        return obs, reward, terminated, truncated, info




In [ ]:
def make_mt5_env():
    return lambda: Monitor(ForexEnv(df_mt5))

env = DummyVecEnv([make_mt5_env()])

## model

In [ ]:
# wrapper qui choisit un start aléatoire avant reset
class RandomStartWrapper(gym.Wrapper):
    def __init__(self, env, min_offset=1, max_offset=None):
        super().__init__(env)
        self.min_offset = min_offset
        self.max_offset = max_offset

    def reset(self, **kwargs):
        max_off = self.max_offset if self.max_offset is not None else max(1, self.env.unwrapped.max_steps // 10)
        start_idx = random.randint(self.min_offset, max_off)
        # positionner current_step puis appeler reset pour générer l'observation
        self.env.unwrapped.current_step = start_idx
        return self.env.reset(**kwargs)

# factory qui retourne env avec random start
def make_mt5_env_random():
    return lambda: Monitor(RandomStartWrapper(ForexEnv(df_mt5), min_offset=1, max_offset=200))

# charger modèle
best_model = PPO.load(MODEL_PATH, env=env)

# éval pré-entrainement non-déterministe
eval_env_pre = DummyVecEnv([make_mt5_env_random()])
pre_rewards, _ = evaluate_policy(best_model, eval_env_pre, n_eval_episodes=10, deterministic=False, return_episode_rewards=True)
print("📊 Évaluation avant entraînement (stochastique):")
print(f"  mean_reward: {np.mean(pre_rewards):.3f}, std: {np.std(pre_rewards):.3f}")
print(f"  episode rewards: {pre_rewards}")

# préparer eval_env pour callback (stochastique)
eval_env = DummyVecEnv([make_mt5_env_random()])

# dossier pour sauvegarder meilleurs modèles durant l'éval
best_eval_dir = "models/eval_mt5"
os.makedirs(best_eval_dir, exist_ok=True)

# EvalCallback en non-deterministic pour refléter la variance réelle
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=best_eval_dir,
    log_path=best_eval_dir,
    eval_freq=10_000,
    n_eval_episodes=5,
    deterministic=False,
    render=False,
)

# entraînement (reprendre)
best_model.learn(
    total_timesteps=RESUME_TIMESTEPS,
    reset_num_timesteps=False,
    callback=eval_callback,
)

# éval post-entrainement non-déterministe
eval_env_post = DummyVecEnv([make_mt5_env_random()])
post_rewards, _ = evaluate_policy(best_model, eval_env_post, n_eval_episodes=10, deterministic=False, return_episode_rewards=True)
print("📊 Évaluation après entraînement (stochastique):")
print(f"  mean_reward: {np.mean(post_rewards):.3f}, std: {np.std(post_rewards):.3f}")
print(f"  episode rewards: {post_rewards}")

# tracer distribution pré / post
plt.figure(figsize=(8,3))
plt.subplot(1,2,1)
plt.hist(pre_rewards, bins=8)
plt.title("Pré entraînement")
plt.subplot(1,2,2)
plt.hist(post_rewards, bins=8)
plt.title("Post entraînement")
plt.tight_layout()
plt.show()

# run manuel (equity-like) pour visualiser pas-à-pas
def run_and_collect(model, env_inst, n_episodes=3):
    all_eps = []
    for _ in range(n_episodes):
        obs, _ = env_inst.reset()
        done = False
        ep_rewards = []
        while not done:
            action, _ = model.predict(obs, deterministic=False)
            obs, reward, terminated, truncated, info = env_inst.step(action)
            ep_rewards.append(float(np.mean(reward) if isinstance(reward, (list, np.ndarray)) else reward))
            done = bool(terminated or truncated)
        all_eps.append(ep_rewards)
    return all_eps

single_env = Monitor(RandomStartWrapper(ForexEnv(df_mt5)))
episodes = run_and_collect(best_model, single_env, n_episodes=20)
# plot equity per episode
plt.figure(figsize=(8,4))
for i, ep in enumerate(episodes):
    plt.plot(np.cumsum(ep), label=f"ep{i+1}")
plt.title("Equity-like (cumsum rewards) - runs stochastiques")
plt.legend()
plt.show()
single_env.close()

# sauvegarder et cleanup
out_path = "models/metatrader_ppo_resumed.zip"
best_model.save(out_path)
print(f"✅ Modèle sauvegardé: {out_path}")
env.close()
eval_env.close()
mt5.shutdown()

In [ ]:
def detect_hallucinations(model, env, n_steps=5000, threshold=0.05):
    """Détecte comportements incohérents — compatible VecEnv et Gymnasium."""
    actions = []
    rewards = []

    # reset (vec env retourne souvent juste obs, non (obs, info))
    reset_res = env.reset()
    obs = reset_res[0] if isinstance(reset_res, tuple) else reset_res

    for _ in range(n_steps):
        action, _ = model.predict(obs, deterministic=False)
        step_res = env.step(action)

        # gérer différentes signatures : (obs, reward, terminated, truncated, info)
        # ou (obs, reward, done, info)
        if len(step_res) == 5:
            obs, reward, terminated, truncated, info = step_res
            done = terminated if isinstance(terminated, (bool, np.bool_)) else np.any(terminated)
            if isinstance(truncated, (bool, np.bool_)):
                done = done or truncated
            else:
                done = done or np.any(truncated)
        elif len(step_res) == 4:
            obs, reward, done, info = step_res
            done = done if isinstance(done, (bool, np.bool_)) else np.any(done)
        else:
            raise ValueError(f"Unexpected env.step return length: {len(step_res)}")

        # stocker action(s) et reward(s)
        actions.append(np.array(action).flatten())
        # reward peut être scalaire ou vecteur (vec env) : prendre la moyenne des envs
        rewards.append(float(np.mean(reward)))

        if done:
            reset_res = env.reset()
            obs = reset_res[0] if isinstance(reset_res, tuple) else reset_res

    actions = np.concatenate([a.reshape(1, -1) for a in actions], axis=0)
    rewards = np.array(rewards)

    # Isolation Forest — adapter reshape si actions multidim
    X = actions.reshape(actions.shape[0], -1)
    od = IForest(threshold=threshold)
    od.fit(X)
    preds = od.predict(X)

    anomaly_ratio = preds["data"]["is_outlier"].mean()
    print("\n=== ANALYSE DES HALLUCINATIONS ===")
    print(f"⚠️ Ratio d'anomalies (risque d'hallucination): {anomaly_ratio:.2%}")

    # distribution des actions (si discret)
    flat_actions = actions.flatten().astype(int)
    action_dist = np.bincount(flat_actions, minlength=int(flat_actions.max() + 1)) / len(flat_actions)
    print("\n📊 Distribution des actions:")
    for i, freq in enumerate(action_dist):
        print(f"  Action {i}: {freq:.1%}")

    print("\n💰 Analyse des rewards:")
    print(f"  Mean: {rewards.mean():.2f}")
    print(f"  Std: {rewards.std():.2f}")
    print(f"  Min: {rewards.min():.2f}")
    print(f"  Max: {rewards.max():.2f}")

    if anomaly_ratio > 0.1:
        print("\n🚨 Possible hallucination détectée!")
    else:
        print("\n✅ Comportement semble stable et cohérent")

    return anomaly_ratio, action_dist, rewards

# Tester les hallucinations
print("\n=== TEST DES HALLUCINATIONS ===")
env_test = DummyVecEnv([make_mt5_env_random()])
anomaly_ratio, action_dist, rewards = detect_hallucinations(best_model, env_test)

# Visualiser la distribution des actions
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.bar(range(len(action_dist)), action_dist)
plt.title("Distribution des actions")
plt.xlabel("Action")
plt.ylabel("Fréquence")

# Visualiser la distribution des rewards
plt.subplot(1,2,2)
plt.hist(rewards, bins=30)
plt.title("Distribution des rewards")
plt.xlabel("Reward")
plt.ylabel("Fréquence")
plt.tight_layout()
plt.show()

env_test.close()

In [ ]:
from stable_baselines3.common.callbacks import EvalCallback
import os

class BestModelSaveCallback(EvalCallback):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.best_mean_reward = -float('inf')
        self.eval_count = 0
        
    def _on_step(self):
        # Appeler méthode parent
        continue_training = super()._on_step()
        
        # Si c'est le moment d'évaluer
        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            self.eval_count += 1
            
            # Calculer reward moyenne
            mean_reward = self.last_mean_reward
            
            # Si meilleure performance, sauvegarder
            if mean_reward > self.best_mean_reward:
                self.best_mean_reward = mean_reward
                
                # Créer nom avec performance
                model_name = f"best_model_reward_{mean_reward:.2f}.zip"
                path = os.path.join(self.best_model_save_path, model_name)
                
                self.model.save(path)
                print(f"\n✨ Nouveau meilleur modèle sauvegardé ({mean_reward:.2f}): {path}")
                
        return continue_training

# Utilisation:
best_eval_dir = "models/eval_mt5"
os.makedirs(best_eval_dir, exist_ok=True)

eval_callback = BestModelSaveCallback(
    eval_env,
    best_model_save_path=best_eval_dir,
    log_path=best_eval_dir,
    eval_freq=10_000,
    n_eval_episodes=5,
    deterministic=False,
    render=False,
)

# Entrainement avec callback personnalisé
best_model.learn(
    total_timesteps=RESUME_TIMESTEPS,
    reset_num_timesteps=False, 
    callback=eval_callback,
)

In [ ]:
from evidently.dashboard import Dashboard
from evidently.dashboard.tabs import DataDriftTab, RegressionPerformanceTab
from evidently import ColumnMapping
import threading
from http.server import HTTPServer, SimpleHTTPRequestHandler

# construire dashboard (réutiliser ref_df_small et curr_df générés précédemment)
dashboard = Dashboard(tabs=[DataDriftTab(), RegressionPerformanceTab()])
col_mapping = ColumnMapping(prediction=None, numerical_features=["price", "delta"], target="reward")
dashboard.calculate(ref_df_small.assign(reward=np.nan), curr_df[["price","delta","reward"]], column_mapping=col_mapping)

# sauvegarder rapport HTML
out_dir = "models/evidently_ui"
os.makedirs(out_dir, exist_ok=True)
report_path = os.path.join(out_dir, "report.html")
dashboard.save(report_path)
print(f"✅ Evidently report saved: {report_path}")

# lancer serveur HTTP simple pour servir le rapport (background thread)
os.chdir(out_dir)
PORT = 8080
handler = SimpleHTTPRequestHandler
httpd = HTTPServer(("0.0.0.0", PORT), handler)

def _serve():
    print(f"📡 Evidently UI disponible sur http://localhost:{PORT}/report.html")
    httpd.serve_forever()

thread = threading.Thread(target=_serve, daemon=True)
thread.start()